In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

with h5py.File(filename, 'r') as f:
    low_cat_matrix = f['data/lowest_tier_category_matrix2'][:]
    #high_cat_matrix = f['data/highest_tier_category_matrix2'][:] # Done later
    arch_gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

arch_category = [x.decode('UTF-8') for x in category_names]
arch_gene_names = [x.decode('UTF-8') for x in arch_gene_names]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]

category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

In [4]:
# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(low_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
low_cat_oe_matrix = np.divide(low_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(low_cat_matrix), where=row_averages[:, np.newaxis]!=0)

low_cat_log2_oe_matrix = np.log2(low_cat_oe_matrix)

---
# Functions

In [5]:
def get_gene_indexes(gene_names):
    gene_indexes = []
    new_gene_names = []
    for gene in gene_names:
        try:
            index = arch_gene_names.index(gene)
            gene_indexes.append(index)
            new_gene_names.append(gene)
        except ValueError:
            pass
    return new_gene_names, gene_indexes

#gene_names, gene_indexes = get_gene_indexes(gene_names)

In [6]:
def get_geneset_category_means(matrix):
    
    # Calculate the column-wise average
    geneset_mean_row = np.mean(matrix, axis=0)

    return geneset_mean_row
    

In [16]:
# Initialize the final dictionary with empty lists for each category
folder = "/home/lwu/ARCHS4/Tests/Check If Biosample Categorys Align With Genesets/"
filename = folder + "h.c1.c2.all.v7.5.1.symbols.gmt"

geneset_means_matrix = []
geneset_names = []
gene_names = []

with open(filename, 'r', ) as file:
    
    for line in file:
#         print(line)
        
        # Extract the geneset name and the corresponding genes 
        line = line.strip().split('\t')
        geneset_names.append(line[0])
        gene_names = line[2:]
#         print(geneset_names, ":", gene_names)
        
        # Match geneset genes to the ARCHS4 gene names list 
        # If gene not found, remove from the gene_names
        # Otherwise, return a list of the genes idsx within ARCHS4
        gene_names, gene_indexes = get_gene_indexes(gene_names)
#         print(gene_names, "\n", gene_indexes)
        
        extracted_matrix = low_cat_log2_oe_matrix[gene_indexes, :] 
#         print(extracted_matrix)
#          print("Extracted Matrix Shape:", extracted_matrix.shape)
        
        geneset_mean_row = get_geneset_category_means(extracted_matrix)
        geneset_means_matrix.append(geneset_mean_row)
        #print("Geneset Matrix Shape:", len(geneset_means_matrix), len(geneset_means_matrix[0]))
#         print()
#         print(geneset_means_matrix)

# View pandas df
#print(geneset_names)
df = pd.DataFrame(geneset_means_matrix, columns=category_list, index=geneset_names)
print(df.shape)
        
df.to_csv('top10genesets_lowest_cat_matrix.csv', index=True)
#display(df)



/home/lwu/.conda/envs/example_env/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/lwu/.conda/envs/example_env/lib/python3.8/site-packages/numpy/core/_methods.py:181: RuntimeWarning: invalid value encountered in true_divide
  ret = um.true_divide(


(6715, 127)


---
# Tried OE

In [8]:
df

,Heart,Atrium,Cardiac Muscle Fiber,Pericardium,Valve,Ventricle,HUVEC,Adipose,Adipocyte,Myofibroblast,...,C42,DU145,LNCAP,PC3,Ovary,Granulosa,Oocyte,ES2,SKOV3,Testis
HALLMARK_TNFA_SIGNALING_VIA_NFKB,-0.612650,-0.600695,NaN,0.313509,-0.243292,-0.867081,-0.269810,-0.171485,-0.331567,NaN,...,-1.886896,-0.675354,-2.195903,-0.392434,-0.292936,0.040357,-2.494972,-0.347550,-0.850586,-1.315508
HALLMARK_HYPOXIA,-0.233649,-0.233237,NaN,-0.256750,-0.293751,-0.373912,-0.625128,-0.175272,-0.022168,NaN,...,-1.177210,-0.746142,-1.572432,-0.637022,-0.309067,-0.104388,-2.004548,-0.243557,-0.723421,-0.761168
HALLMARK_CHOLESTEROL_HOMEOSTASIS,-0.466517,-0.598194,NaN,-0.643731,-0.412744,-0.689433,-0.467987,0.005675,0.288194,NaN,...,-0.333905,-0.222263,-0.506447,-0.182145,-0.301905,0.256869,-1.715743,-0.002714,-0.326528,-0.427872
HALLMARK_MITOTIC_SPINDLE,-0.365007,-0.798872,NaN,-0.811466,-0.130742,-0.882279,0.083652,-0.589819,-0.369502,NaN,...,-0.377425,0.018118,-0.487955,-0.049084,-0.076599,-0.517167,-0.621121,-0.000875,0.033729,-0.157238
HALLMARK_WNT_BETA_CATENIN_SIGNALING,-0.253789,-0.537643,NaN,-0.271314,-0.334614,-0.555743,-0.463658,-0.368907,-0.245804,NaN,...,-0.893482,-0.846051,-0.933728,-0.467696,-0.076474,-0.446853,-1.044369,-0.107774,-0.665238,-0.273549
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SA_MMP_CYTOKINE_CONNECTION,-0.760974,-0.972834,NaN,0.668151,-0.290723,-1.072619,-1.149949,-0.167377,-1.406919,NaN,...,-3.052047,-1.758735,-3.986742,-1.417438,-0.655049,-0.647243,-4.386416,-0.806157,-1.915436,-2.080514
SA_PROGRAMMED_CELL_DEATH,-0.488302,-0.679560,NaN,-0.414894,-0.411818,-0.591173,0.003343,-0.366873,-0.270821,NaN,...,-0.753881,-0.336032,-0.586390,-0.140101,-0.002965,-0.275475,-1.316587,-0.134749,-0.347372,-0.677755
SA_PTEN_PATHWAY,-0.422671,-0.333453,NaN,-0.026631,-0.270148,-0.492782,-0.320633,-0.101259,-0.168125,NaN,...,-0.258166,-0.362807,-0.311574,-0.430144,-0.173076,-0.310511,-0.798396,-0.335572,-0.569925,-0.673403
SA_REG_CASCADE_OF_CYCLIN_EXPR,-1.392220,-2.264685,NaN,-2.655673,-0.891104,-2.129027,-0.149196,-1.553106,-0.797230,NaN,...,-0.326095,0.533928,-0.295041,0.270735,-0.121407,-1.197322,-1.907545,0.030757,0.219907,-0.692965


In [9]:
# loop through each column and get the top 10 highest values and their indexes
for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Heart":
BRUNEAU_SEPTATION_ATRIAL                                  3.935864
REACTOME_PHYSIOLOGICAL_FACTORS                            2.103059
CHEMELLO_SOLEUS_VS_EDL_MYOFIBERS_UP                       1.902356
WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY                    1.770818
REACTOME_STRIATED_MUSCLE_CONTRACTION                      1.601546
RICKMAN_HEAD_AND_NECK_CANCER_F                            1.497078
BRUNEAU_SEPTATION_VENTRICULAR                             1.450855
CASTELLANO_HRAS_AND_NRAS_TARGETS_DN                       1.354382
PLASARI_TGFB1_TARGETS_1HR_DN                              1.302444
REACTOME_YAP1_AND_WWTR1_TAZ_STIMULATED_GENE_EXPRESSION    1.220103
Name: Heart, dtype: float64
Indexes for Column "Heart":
Index(['BRUNEAU_SEPTATION_ATRIAL', 'REACTOME_PHYSIOLOGICAL_FACTORS',
       'CHEMELLO_SOLEUS_VS_EDL_MYOFIBERS_UP',
       'WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY',
       'REACTOME_STRIATED_MUSCLE_CONTRACTION',
       'RICKMAN_HEAD_AND_NECK_

Top 10 values in Column "T47D":
REACTOME_SIGNALING_BY_MST1                                   1.594549
FARMER_BREAST_CANCER_CLUSTER_6                               1.502527
YANG_BREAST_CANCER_ESR1_UP                                   1.206411
FARMER_BREAST_CANCER_CLUSTER_7                               1.132371
LIEN_BREAST_CARCINOMA_METAPLASTIC_VS_DUCTAL_DN               1.121113
REACTOME_BETA_OXIDATION_OF_VERY_LONG_CHAIN_FATTY_ACIDS       1.028233
CHARAFE_BREAST_CANCER_LUMINAL_VS_MESENCHYMAL_UP              1.012835
MASRI_RESISTANCE_TO_TAMOXIFEN_AND_AROMATASE_INHIBITORS_UP    0.964989
HOLLERN_EMT_BREAST_TUMOR_DN                                  0.896757
HUANG_DASATINIB_RESISTANCE_DN                                0.885536
Name: T47D, dtype: float64
Indexes for Column "T47D":
Index(['REACTOME_SIGNALING_BY_MST1', 'FARMER_BREAST_CANCER_CLUSTER_6',
       'YANG_BREAST_CANCER_ESR1_UP', 'FARMER_BREAST_CANCER_CLUSTER_7',
       'LIEN_BREAST_CARCINOMA_METAPLASTIC_VS_DUCTAL_DN',
       'REACTOM

---
# Below is Log2(OE)

In [10]:
df

,Heart,Atrium,Cardiac Muscle Fiber,Pericardium,Valve,Ventricle,HUVEC,Adipose,Adipocyte,Myofibroblast,...,C42,DU145,LNCAP,PC3,Ovary,Granulosa,Oocyte,ES2,SKOV3,Testis
HALLMARK_TNFA_SIGNALING_VIA_NFKB,-0.612650,-0.600695,NaN,0.313509,-0.243292,-0.867081,-0.269810,-0.171485,-0.331567,NaN,...,-1.886896,-0.675354,-2.195903,-0.392434,-0.292936,0.040357,-2.494972,-0.347550,-0.850586,-1.315508
HALLMARK_HYPOXIA,-0.233649,-0.233237,NaN,-0.256750,-0.293751,-0.373912,-0.625128,-0.175272,-0.022168,NaN,...,-1.177210,-0.746142,-1.572432,-0.637022,-0.309067,-0.104388,-2.004548,-0.243557,-0.723421,-0.761168
HALLMARK_CHOLESTEROL_HOMEOSTASIS,-0.466517,-0.598194,NaN,-0.643731,-0.412744,-0.689433,-0.467987,0.005675,0.288194,NaN,...,-0.333905,-0.222263,-0.506447,-0.182145,-0.301905,0.256869,-1.715743,-0.002714,-0.326528,-0.427872
HALLMARK_MITOTIC_SPINDLE,-0.365007,-0.798872,NaN,-0.811466,-0.130742,-0.882279,0.083652,-0.589819,-0.369502,NaN,...,-0.377425,0.018118,-0.487955,-0.049084,-0.076599,-0.517167,-0.621121,-0.000875,0.033729,-0.157238
HALLMARK_WNT_BETA_CATENIN_SIGNALING,-0.253789,-0.537643,NaN,-0.271314,-0.334614,-0.555743,-0.463658,-0.368907,-0.245804,NaN,...,-0.893482,-0.846051,-0.933728,-0.467696,-0.076474,-0.446853,-1.044369,-0.107774,-0.665238,-0.273549
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SA_MMP_CYTOKINE_CONNECTION,-0.760974,-0.972834,NaN,0.668151,-0.290723,-1.072619,-1.149949,-0.167377,-1.406919,NaN,...,-3.052047,-1.758735,-3.986742,-1.417438,-0.655049,-0.647243,-4.386416,-0.806157,-1.915436,-2.080514
SA_PROGRAMMED_CELL_DEATH,-0.488302,-0.679560,NaN,-0.414894,-0.411818,-0.591173,0.003343,-0.366873,-0.270821,NaN,...,-0.753881,-0.336032,-0.586390,-0.140101,-0.002965,-0.275475,-1.316587,-0.134749,-0.347372,-0.677755
SA_PTEN_PATHWAY,-0.422671,-0.333453,NaN,-0.026631,-0.270148,-0.492782,-0.320633,-0.101259,-0.168125,NaN,...,-0.258166,-0.362807,-0.311574,-0.430144,-0.173076,-0.310511,-0.798396,-0.335572,-0.569925,-0.673403
SA_REG_CASCADE_OF_CYCLIN_EXPR,-1.392220,-2.264685,NaN,-2.655673,-0.891104,-2.129027,-0.149196,-1.553106,-0.797230,NaN,...,-0.326095,0.533928,-0.295041,0.270735,-0.121407,-1.197322,-1.907545,0.030757,0.219907,-0.692965


---
# TOP GENESETS

In [11]:
# loop through each column and get the top 10 highest values and their indexes
for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    #print('Indexes for Column "{}":'.format(col))
    #print(top_10.index)
    print()

Top 10 values in Column "Heart":
BRUNEAU_SEPTATION_ATRIAL                                  3.935864
REACTOME_PHYSIOLOGICAL_FACTORS                            2.103059
CHEMELLO_SOLEUS_VS_EDL_MYOFIBERS_UP                       1.902356
WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY                    1.770818
REACTOME_STRIATED_MUSCLE_CONTRACTION                      1.601546
RICKMAN_HEAD_AND_NECK_CANCER_F                            1.497078
BRUNEAU_SEPTATION_VENTRICULAR                             1.450855
CASTELLANO_HRAS_AND_NRAS_TARGETS_DN                       1.354382
PLASARI_TGFB1_TARGETS_1HR_DN                              1.302444
REACTOME_YAP1_AND_WWTR1_TAZ_STIMULATED_GENE_EXPRESSION    1.220103
Name: Heart, dtype: float64

Top 10 values in Column "Atrium":
BRUNEAU_SEPTATION_ATRIAL                                              4.470016
REACTOME_PHYSIOLOGICAL_FACTORS                                        2.542300
CHEMELLO_SOLEUS_VS_EDL_MYOFIBERS_UP                                   1.7672

Top 10 values in Column "Sensory Neuron":
REACTOME_TOXICITY_OF_BOTULINUM_TOXIN_TYPE_D_BOTD                                    3.486561
REACTOME_NEUROTOXICITY_OF_CLOSTRIDIUM_TOXINS                                        3.240493
REACTOME_MUSCARINIC_ACETYLCHOLINE_RECEPTORS                                         2.644046
REACTOME_HIGHLY_CALCIUM_PERMEABLE_NICOTINIC_ACETYLCHOLINE_RECEPTORS                 2.643052
REACTOME_LGI_ADAM_INTERACTIONS                                                      2.634602
REACTOME_PRESYNAPTIC_DEPOLARIZATION_AND_CALCIUM_CHANNEL_OPENING                     2.364301
REACTOME_HIGHLY_CALCIUM_PERMEABLE_POSTSYNAPTIC_NICOTINIC_ACETYLCHOLINE_RECEPTORS    2.284138
REACTOME_SEROTONIN_NEUROTRANSMITTER_RELEASE_CYCLE                                   2.179410
REACTOME_NEUROFASCIN_INTERACTIONS                                                   2.172124
LE_NEURONAL_DIFFERENTIATION_UP                                                      2.161370
Name: Sensory Neuron, dtype:

Top 10 values in Column "PC3":
NIKOLSKY_BREAST_CANCER_10Q22_AMPLICON                 1.056518
REACTOME_EGFR_INTERACTS_WITH_PHOSPHOLIPASE_C_GAMMA    0.774232
REACTOME_SENSORY_PERCEPTION_OF_SALTY_TASTE            0.749992
FOURNIER_ACINAR_DEVELOPMENT_LATE_DN                   0.722655
KOBAYASHI_EGFR_SIGNALING_6HR_DN                       0.718929
REACTOME_SIGNALING_BY_RNF43_MUTANTS                   0.677309
REACTOME_TYPE_I_HEMIDESMOSOME_ASSEMBLY                0.622133
JI_CARCINOGENESIS_BY_KRAS_AND_STK11_UP                0.592528
GLINSKY_CANCER_DEATH_UP                               0.590835
LOPEZ_MESOTELIOMA_SURVIVAL_TIME_UP                    0.566029
Name: PC3, dtype: float64

Top 10 values in Column "Ovary":
TESAR_ALK_AND_JAK_TARGETS_MOUSE_ES_D4_DN                                        1.930605
TESAR_ALK_TARGETS_HUMAN_ES_5D_DN                                                1.641570
SALVADOR_MARTIN_PEDIATRIC_TBD_ANTI_TNF_THERAPY_NONRESPONDER_PRE_TREATMENT_UP    1.637023
TESAR_ALK_TA